In [23]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# os.environ["JAX_ENABLE_X64"] = "true" # TODO

from typing import Callable, Literal

import numpy as onp
import jax
import jax.numpy as jnp
from jax.typing import ArrayLike
from jax import Array

from msmjax.convenience import (
    set_up_kernels_grids_and_stencils,
    suggest_msm_params,
)
from msmjax.kernels import _compute_kernel_stencil
from msmjax.benchmark_tools import path_input_structures
from msmjax.bspline_interpolation.coefficients import (
    compute_coeffs_with_truncation,
)

# Function definitions

In [123]:
# TODO: define elsewhere
CellMode = Literal["ortho", "general"]

from msmjax.utils import _sqrt


def get_distances(sizes_from_center, spacing_or_gridcell):
    # TODO: jnp?
    indices_per_axis = [onp.arange(-s, s + 1) for s in sizes_from_center]
    indices = onp.stack(
        onp.meshgrid(*indices_per_axis, indexing="ij"), axis=-1
    )
    if onp.ndim(spacing_or_gridcell) < 2:
        points = indices * spacing_or_gridcell
    else:
        points = indices @ spacing_or_gridcell
    return _sqrt((points * points).sum(axis=-1))


def _compute_one_stencil(values: ArrayLike, omega: ArrayLike, mode: str):
    def _conv_1d(in1: ArrayLike, in2: ArrayLike):
        # TODO: method="fft"?
        return jax.scipy.signal.convolve(in1, in2, mode=mode)

    result = values
    for axis in range(values.ndim):
        result = jnp.apply_along_axis(
            func1d=_conv_1d, axis=axis, arr=result, in2=omega
        )
    return result


def construct_stencils(
    omega: ArrayLike,
    n_levels_intermediate: int,
    include_toplevel: bool,
    k_lvl_1: Callable[[ArrayLike], Array] = None,
    sizes_intermediate: tuple[int, ...] = None,
    spacings_or_gridcell_lvl_1: ArrayLike = None,
    k_toplevel: Callable[[ArrayLike], Array] = None,
    grid_shape_toplevel: tuple[int, ...] = None,
) -> list[Array]:
    # Placeholder for level zero (l = 0), at which there is no grid:
    stencils = [None]

    if n_levels_intermediate < 0:
        raise ValueError("n_levels_intermediate must be >= 0")
    if n_levels_intermediate == 0 and not include_toplevel:
        raise ValueError(
            "n_levels_intermediate = 0 and include_toplevel = False "
            "at the same is not allowed (this would mean that "
            "there isn't a single grid level)."
        )
    args_intermediate = [
        k_lvl_1,
        sizes_intermediate,
        spacings_or_gridcell_lvl_1,
    ]
    if n_levels_intermediate > 0 and any(
        [x is None for x in args_intermediate]
    ):
        raise ValueError(
            "k_lvl_1, sizes_intermediate, spacings_or_gridcell_lvl_1 "
            "are required when n_levels_intermediate > 0."
        )
    args_toplevel = [k_toplevel, grid_shape_toplevel]
    if include_toplevel and any([x is None for x in args_toplevel]):
        raise ValueError(
            "k_toplevel, grid_shape_toplevel "
            "are required when include_toplevel = True."
        )

    # Intermediate levels (l = 1 ... L - 1):
    if n_levels_intermediate > 0:
        distances_lvl_1 = get_distances(
            sizes_intermediate, spacings_or_gridcell_lvl_1
        )
        kernel_values_at_gridpoints = k_lvl_1(distances_lvl_1)
        stencils.append(
            _compute_one_stencil(
                kernel_values_at_gridpoints, omega, mode="same"
            )
        )
        for lvl in range(n_levels_intermediate - 1):
            stencils.append(0.5 * stencils[-1])

    # Top level containing long-range tail (l = L), if included
    if include_toplevel:
        # TODO: explain why the sizes are the way they are
        sizes_toplevel = tuple(
            (s - 1) + len(omega) // 2 for s in grid_shape_toplevel
        )
        max_grid_level = n_levels_intermediate + 1
        distances_toplevel = get_distances(
            sizes_toplevel,
            2 ** (max_grid_level - 1) * spacings_or_gridcell_lvl_1,
        )
        kernel_values_at_gridpoints = k_toplevel(distances_toplevel)
        stencils.append(
            _compute_one_stencil(
                kernel_values_at_gridpoints, omega, mode="valid"
            )
        )

    return stencils


def make_dynamic_cell_construct_stencils(scaled_spacings, cell_mode):
    def dynamic_construct_stencils(cell):
        pass  # TODO

    return dynamic_construct_stencils

# Get structure

In [95]:
structures = onp.load(path_input_structures / "structures_10000.npz")

pos = structures["positions"][0]
chg = structures["charges"][0]
cll = structures["cells"][0]

(n_particles, n_dim) = pos.shape
box_lengths = onp.diag(cll)

In [96]:
PBC = (False, False, False)
LEVEL_ONE_GRIDSPACING = 1.0
LEVEL_ZERO_CUTOFF = 3.0
P = 4

In [97]:
params = suggest_msm_params(
    box_lengths=box_lengths,
    pbc=PBC,
    n_particles=n_particles,
    level_one_gridspacing=LEVEL_ONE_GRIDSPACING,
    level_zero_cutoff=LEVEL_ZERO_CUTOFF,
    p=P,
)
kernel_fns, grids, stencils_old = set_up_kernels_grids_and_stencils(
    box_lengths=box_lengths,
    pbc=PBC,
    **params,
)
omega, _ = compute_coeffs_with_truncation(params["p"], params["mu"])

Suggested value for max grid level L was found according to criterion/criteria:
(if multiple criteria listed, that means they agree on the suggested L)
*) Highest-level grid not coarser than simulation box size ("criterion 1.")



In [98]:
grid_shapes = [None if x is None else x.shape for x in grids]
grid_shapes

[None, (27, 27, 27), (16, 16, 16), (11, 11, 11), (8, 8, 8), (7, 7, 7)]

In [105]:
stencil_shapes_old = [None if x is None else x.shape for x in stencils_old]
stencil_shapes_old

[None, (17, 17, 17), (17, 17, 17), (17, 17, 17), (17, 17, 17), (47, 47, 47)]

In [124]:
alpha = int(
    (
        params["level_zero_cutoff"]
        / onp.asarray(params["level_one_gridspacing"])
    ).max()
)
# TODO: Does it need to be +2 or is +1 enough?
sizes_intermediate = onp.full_like(box_lengths, 2 * alpha + 1, dtype=int)

sizes_toplevel = tuple(s + len(omega) // 2 for s in grids[-1].shape)

In [125]:
sizes_toplevel

(23, 23, 23)

In [127]:
stencils_new = construct_stencils(
    omega=omega,
    n_levels_intermediate=len(kernel_fns) - 2,
    include_toplevel=True,  # TODO: pbc-dependent
    k_lvl_1=kernel_fns[1],
    sizes_intermediate=sizes_intermediate,
    spacings_or_gridcell_lvl_1=onp.asarray(params["level_one_gridspacing"]),
    k_toplevel=kernel_fns[-1],  # TODO: pbc-dependent
    grid_shape_toplevel=grid_shapes[-1],  # TODO: pbc-dependent
)

In [128]:
stencil_shapes_new = [None if x is None else x.shape for x in stencils_new]
stencil_shapes_new

[None, (15, 15, 15), (15, 15, 15), (15, 15, 15), (15, 15, 15), (13, 13, 13)]

In [134]:
def trim_array_centered(arr, target_shape):
    shape = onp.array(arr.shape)
    target_shape = onp.asarray(target_shape)
    assert (shape >= target_shape).all()
    excess_lengths_per_axis = shape - target_shape
    return arr[tuple(slice(e, -e) for e in excess_lengths_per_axis // 2)]


for s_new, s_old in zip(stencils_new[1:], stencils_old[1:]):
    s_old_trimmed = trim_array_centered(s_old, s_new.shape)
    assert onp.allclose(s_new, s_old_trimmed)

In [135]:
grid_shapes

[None, (27, 27, 27), (16, 16, 16), (11, 11, 11), (8, 8, 8), (7, 7, 7)]

In [136]:
trim_per_axis = (
    onp.asarray(stencil_shapes_old[-1])
    - (2 * onp.asarray(grid_shapes[-1]) - 1)
) // 2

In [119]:
trim_per_axis

array([17, 17, 17])

In [120]:
trimmed_toplevel_stencil_old = stencils_old[-1][
    tuple(slice(t, -t) for t in trim_per_axis)
]

In [121]:
trimmed_toplevel_stencil_old.shape

(13, 13, 13)

In [122]:
# onp.allclose(stencils_new[-1][1:-1, 1:-1, 1:-1], trimmed_toplevel_stencil_old)
onp.allclose(stencils_new[-1], trimmed_toplevel_stencil_old)

True